# RNNs and LSTMs: Mathematical Foundations

> *"The LSTM is one of the most important contributions in the history of neural networks — not because it's complicated, but because it's exactly complicated enough."*

Interactive theory notebook. Run cells top-to-bottom. Each section maps directly to the approved TOC in `notes.md`.

In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    import matplotlib
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [11, 6]
    plt.rcParams['font.size'] = 12
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print('matplotlib not available — skipping plots')

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)
print('Setup complete.')

---

## 1. Intuition: Sequences and the Memory Bottleneck

A feedforward network treats each input independently. For sequences, we need to carry context across steps via a **hidden state** $\mathbf{h}_t \in \mathbb{R}^d$.

Below we visualise the **information decay** problem: how quickly does a signal injected at step 0 influence the hidden state at step $t$?

In [ ]:
# === 1.1 Information Decay with Spectral Radius ===
# How quickly does h_0 influence h_T as T grows?

d = 10  # hidden dim
T = 60  # sequence length

def make_rnn_Wh(d, spectral_radius, seed=42):
    """Create W_h with prescribed spectral radius via orthogonal matrix."""
    rng = np.random.default_rng(seed)
    Q, _ = la.qr(rng.standard_normal((d, d)))
    return spectral_radius * Q

radii = [0.5, 0.9, 0.99, 1.0]
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, rho in enumerate(radii):
    Wh = make_rnn_Wh(d, rho)
    norms = []
    Wh_power = np.eye(d)
    for t in range(1, T+1):
        Wh_power = Wh_power @ Wh
        norms.append(la.norm(Wh_power, 'fro'))
    if HAS_MPL:
        axes[0].plot(range(1, T+1), norms, label=f'rho={rho}', color=colors[i])
    print(f'rho={rho}: norm(Wh^50) = {norms[49]:.4f}')

if HAS_MPL:
    axes[0].set_xlabel('Time step T')
    axes[0].set_ylabel('||W_h^T||_F')
    axes[0].set_title('Information Decay: ||W_h^T||_F vs T')
    axes[0].legend()
    axes[0].set_yscale('log')

    # Phase portrait of a 2D linear RNN
    Wh2 = make_rnn_Wh(2, 0.95)
    theta = np.linspace(0, 2*np.pi, 12)
    for th in theta:
        h = np.array([np.cos(th), np.sin(th)])
        traj = [h.copy()]
        for _ in range(30):
            h = Wh2 @ h
            traj.append(h.copy())
        traj = np.array(traj)
        axes[1].plot(traj[:,0], traj[:,1], alpha=0.5, linewidth=0.8)
    axes[1].set_title('Phase portrait: rho=0.95 (spiral to origin)')
    axes[1].set_xlabel('h_1'); axes[1].set_ylabel('h_2')
    plt.tight_layout()
    plt.show()
    print('Figure: information decay (log scale) and 2D phase portrait')

---

## 2. Vanilla RNN: Forward Pass

The full recurrence:
$$\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b}_h)$$
$$\hat{\mathbf{y}}_t = \text{softmax}(W_o \mathbf{h}_t + \mathbf{b}_o)$$

We implement this from scratch and verify against a naive loop.

In [ ]:
# === 2.1 Vanilla RNN Forward Pass ===

class VanillaRNN:
    def __init__(self, input_dim, hidden_dim, output_dim, seed=42):
        rng = np.random.default_rng(seed)
        scale = 1.0 / np.sqrt(hidden_dim)
        self.Wh = rng.standard_normal((hidden_dim, hidden_dim)) * scale
        self.Wx = rng.standard_normal((hidden_dim, input_dim)) * scale
        self.bh = np.zeros(hidden_dim)
        self.Wo = rng.standard_normal((output_dim, hidden_dim)) * scale
        self.bo = np.zeros(output_dim)
        self.hidden_dim = hidden_dim

    def forward(self, xs):
        """xs: (T, input_dim) -> (hs: (T, d), ys: (T, output_dim))"""
        T = len(xs)
        h = np.zeros(self.hidden_dim)
        hs, ys = [], []
        for t in range(T):
            h = np.tanh(self.Wh @ h + self.Wx @ xs[t] + self.bh)
            z = self.Wo @ h + self.bo
            y = np.exp(z) / np.exp(z).sum()  # softmax
            hs.append(h.copy())
            ys.append(y.copy())
        return np.array(hs), np.array(ys)

# Demo
p, d, k, T = 8, 16, 5, 10
rnn = VanillaRNN(p, d, k)
xs = np.random.randn(T, p)
hs, ys = rnn.forward(xs)

print(f'Input shape:  {xs.shape}')
print(f'Hidden shape: {hs.shape}')
print(f'Output shape: {ys.shape}')
print(f'h_T norm: {la.norm(hs[-1]):.4f}  (should be < 1 due to tanh)')
print(f'y_T sum:  {ys[-1].sum():.8f}  (should be 1.0 — softmax)')
assert np.allclose(ys[-1].sum(), 1.0), 'softmax must sum to 1'
print('PASS — softmax outputs sum to 1')

# Verify: W_h = 0 means no memory
rnn_nomem = VanillaRNN(p, d, k)
rnn_nomem.Wh[:] = 0.0
hs2, _ = rnn_nomem.forward(xs)
print(f'\nWith Wh=0: all hidden states identical given same x?')
# With Wh=0, each h_t depends only on x_t, not h_{t-1}
h_from_x1 = np.tanh(rnn_nomem.Wx @ xs[0] + rnn_nomem.bh)
print(f'PASS — Wh=0 reduces to memoryless MLP: {np.allclose(hs2[0], h_from_x1)}')

In [ ]:
# === 2.3 Parameter Count and Weight Sharing ===

def rnn_param_count(p, d, k, tied_embedding=False):
    """Count parameters in a vanilla RNN."""
    Wh = d * d
    Wx = d * p
    bh = d
    Wo = 0 if tied_embedding else k * d
    bo = k
    total = Wh + Wx + bh + Wo + bo
    return {'Wh': Wh, 'Wx': Wx, 'bh': bh, 'Wo': Wo, 'bo': bo, 'total': total}

configs = [(64, 256, 50), (512, 512, 10000), (1024, 2048, 50000)]
print(f'{'Config':<30} {'Total params':>15} {'Independent of T?':>18}')
print('-' * 65)
for p, d, k in configs:
    counts = rnn_param_count(p, d, k)
    label = f'p={p}, d={d}, k={k}'
    print(f'{label:<30} {counts["total"]:>15,}  {'YES — shared across steps':>18}')

print('\nKey insight: params scale as O(d^2 + dp + kd), NOT O(T)')
print('A transformer\'s KV cache scales as O(T * d) — linear in T')
print('An RNN\'s inference memory scales as O(d) — constant in T')

---

## 3. Backpropagation Through Time (BPTT)

The gradient of the loss at step $T$ with respect to hidden state $\mathbf{h}_k$ involves a product of Jacobians:
$$\frac{\partial \mathbf{h}_T}{\partial \mathbf{h}_k} = \prod_{j=k}^{T-1} \underbrace{D_j W_h}_{J_j}$$
where $D_j = \mathrm{diag}(\sigma'(\mathbf{z}_{j+1}))$. When $\rho(W_h) < 1$, this product decays exponentially — **vanishing gradients**.

In [ ]:
# === 3.2 Numerical BPTT Gradient Verification ===
# Compare analytic BPTT gradient to finite-difference numerical gradient

def rnn_loss(Wh, Wx, bh, xs, targets):
    """Cross-entropy loss for a scalar-output RNN."""
    T = len(xs)
    d = Wh.shape[0]
    h = np.zeros(d)
    loss = 0.0
    for t in range(T):
        h = np.tanh(Wh @ h + Wx @ xs[t] + bh)
        # single linear output for simplicity
        y_hat = h[0]  # predict first hidden unit
        loss += 0.5 * (y_hat - targets[t]) ** 2
    return loss

# Small network for numerical verification
p2, d2, T2 = 3, 4, 5
rng = np.random.default_rng(0)
Wh = rng.standard_normal((d2, d2)) * 0.5
Wx = rng.standard_normal((d2, p2)) * 0.5
bh = np.zeros(d2)
xs2 = rng.standard_normal((T2, p2))
targets = rng.standard_normal(T2) * 0.1

# Numerical gradient via finite differences
eps = 1e-5
grad_Wh_numerical = np.zeros_like(Wh)
for i in range(d2):
    for j in range(d2):
        Wh_p = Wh.copy(); Wh_p[i, j] += eps
        Wh_m = Wh.copy(); Wh_m[i, j] -= eps
        grad_Wh_numerical[i, j] = (rnn_loss(Wh_p, Wx, bh, xs2, targets) -
                                   rnn_loss(Wh_m, Wx, bh, xs2, targets)) / (2 * eps)

# Analytic gradient via manual BPTT
def rnn_bptt(Wh, Wx, bh, xs, targets):
    T = len(xs); d = Wh.shape[0]
    hs = [np.zeros(d)]
    zs = []
    for t in range(T):
        z = Wh @ hs[-1] + Wx @ xs[t] + bh
        zs.append(z)
        hs.append(np.tanh(z))
    # Backward
    dh = np.zeros(d)
    dWh = np.zeros_like(Wh)
    for t in reversed(range(T)):
        dy = hs[t+1][0] - targets[t]  # loss gradient w.r.t. output
        dh_from_loss = np.zeros(d); dh_from_loss[0] = dy
        dh = dh + dh_from_loss
        dtanh = (1 - hs[t+1] ** 2)  # tanh derivative
        dz = dtanh * dh
        dWh += np.outer(dz, hs[t])
        dh = Wh.T @ dz
    return dWh

grad_Wh_analytic = rnn_bptt(Wh, Wx, bh, xs2, targets)
max_err = np.max(np.abs(grad_Wh_analytic - grad_Wh_numerical))
print(f'Max absolute error (analytic vs numerical): {max_err:.2e}')
ok = max_err < 1e-5
print(f'{'PASS' if ok else 'FAIL'} — BPTT gradient matches finite differences')

In [ ]:
# === 3.3 Vanishing Gradients: Jacobian Product Norm ===

T_max = 80
d3 = 20

results = {}
for rho in [0.5, 0.9, 1.0, 1.05]:
    Q, _ = la.qr(np.random.randn(d3, d3))
    Wh3 = rho * Q
    # Simulate gradient norm: at each step, multiply by D_t W_h
    # Use D_t = identity (max case) for upper bound
    norms = []
    J_prod = np.eye(d3)
    for t in range(T_max):
        # Random D_t for tanh: diagonal entries in (0,1)
        h_t = np.tanh(np.random.randn(d3))
        D = np.diag(1 - h_t**2)
        J_prod = D @ Wh3 @ J_prod
        norms.append(la.norm(J_prod, 'fro'))
    results[rho] = norms

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 5))
    colors2 = ['#d62728', '#ff7f0e', '#2ca02c', '#9467bd']
    for (rho, norms), c in zip(results.items(), colors2):
        ax.semilogy(range(1, T_max+1), [max(n, 1e-15) for n in norms],
                    label=f'rho={rho}', color=c)
    ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.4, label='norm=1')
    ax.set_xlabel('Number of time steps back')
    ax.set_ylabel('||Jacobian product||_F (log scale)')
    ax.set_title('Vanishing/Exploding Gradients vs Spectral Radius')
    ax.legend()
    plt.tight_layout()
    plt.show()

print(f'At T=50 steps:')
for rho, norms in results.items():
    print(f'  rho={rho}: gradient norm = {norms[49]:.2e}')

In [ ]:
# === 3.4 Gradient Clipping ===

def clip_gradient(params_grad, threshold):
    """Global gradient norm clipping."""
    total_norm = np.sqrt(sum(la.norm(g)**2 for g in params_grad))
    if total_norm > threshold:
        factor = threshold / total_norm
        params_grad = [g * factor for g in params_grad]
    return params_grad, total_norm

# Demonstrate clipping preserves direction
rng2 = np.random.default_rng(7)
g1 = rng2.standard_normal((50, 50)) * 100  # large gradient
g2 = rng2.standard_normal(50) * 100
tau = 5.0

[g1_c, g2_c], total_before = clip_gradient([g1, g2], tau)
total_after = np.sqrt(la.norm(g1_c)**2 + la.norm(g2_c)**2)

print(f'Gradient norm BEFORE clipping: {total_before:.2f}')
print(f'Gradient norm AFTER  clipping: {total_after:.2f}  (threshold: {tau})')

# Direction preserved?
cos_sim = np.dot(g1.ravel(), g1_c.ravel()) / (la.norm(g1) * la.norm(g1_c))
print(f'Cosine similarity (direction preserved): {cos_sim:.8f}')
print(f'PASS — direction preserved: {np.isclose(cos_sim, 1.0, atol=1e-6)}')
print(f'\nClipping inactive when norm < threshold:')
small_g = [np.ones((3, 3))]
[small_g_c], norm_small = clip_gradient(small_g, tau)
print(f'  norm={norm_small:.2f} < {tau}: unchanged = {np.allclose(small_g[0], small_g_c[0])}')

---

## 4. Long Short-Term Memory (LSTM)

The LSTM introduces two states:
- **Cell state** $\mathbf{c}_t$: long-term memory, updated **additively**
- **Hidden state** $\mathbf{h}_t$: short-term working representation

Four gate equations:
$$\mathbf{f}_t = \sigma(W_f [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_f)$$
$$\mathbf{i}_t = \sigma(W_i [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_i)$$
$$\tilde{\mathbf{c}}_t = \tanh(W_c [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_c)$$
$$\mathbf{o}_t = \sigma(W_o [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_o)$$
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

In [ ]:
# === 4.2 LSTM Forward Pass from Scratch ===

class LSTMCell:
    def __init__(self, input_dim, hidden_dim, seed=42):
        rng = np.random.default_rng(seed)
        n = hidden_dim + input_dim
        scale = 1.0 / np.sqrt(n)
        # Stacked weight: [W_f; W_i; W_c; W_o] shape = (4d, d+p)
        self.W = rng.standard_normal((4 * hidden_dim, n)) * scale
        self.b = np.zeros(4 * hidden_dim)
        # Forget gate bias = 1 (best practice)
        self.b[:hidden_dim] = 1.0
        self.d = hidden_dim

    def step(self, h_prev, c_prev, x):
        d = self.d
        inp = np.concatenate([h_prev, x])
        gates = self.W @ inp + self.b
        f = 1 / (1 + np.exp(-gates[0*d:1*d]))    # forget gate
        i = 1 / (1 + np.exp(-gates[1*d:2*d]))    # input gate
        c_tilde = np.tanh(gates[2*d:3*d])         # candidate cell
        o = 1 / (1 + np.exp(-gates[3*d:4*d]))    # output gate
        c = f * c_prev + i * c_tilde              # cell state update
        h = o * np.tanh(c)                        # hidden state
        return h, c, {'f': f, 'i': i, 'c_tilde': c_tilde, 'o': o}

    def forward(self, xs):
        T, p = xs.shape
        h, c = np.zeros(self.d), np.zeros(self.d)
        hs, cs, gates_list = [], [], []
        for t in range(T):
            h, c, g = self.step(h, c, xs[t])
            hs.append(h.copy()); cs.append(c.copy()); gates_list.append(g)
        return np.array(hs), np.array(cs), gates_list

# Demo
p4, d4, T4 = 8, 12, 15
lstm = LSTMCell(p4, d4)
xs4 = np.random.randn(T4, p4)
hs4, cs4, gs4 = lstm.forward(xs4)

print(f'LSTM output shapes: h={hs4.shape}, c={cs4.shape}')
print(f'h_T range: [{hs4[-1].min():.3f}, {hs4[-1].max():.3f}]  (bounded by tanh)')
print(f'c_T range: [{cs4[-1].min():.3f}, {cs4[-1].max():.3f}]  (unbounded cell)')
print(f'Forget gate t=5: {gs4[4]["f"][:4].round(3)}  (near 0.73 due to bias=1 init)')
print(f'Input gate t=5:  {gs4[4]["i"][:4].round(3)}')
print(f'Output gate t=5: {gs4[4]["o"][:4].round(3)}')

In [ ]:
# === 4.3 Cell State Highway: Gradient Through c_t ===
# Show that gradient of c_T w.r.t. c_0 is approximately identity when f_t = 1

d5 = 8
T5 = 20

def cell_grad_through_forget(forget_value, T, d):
    """Compute ||d c_T / d c_0|| when all forget gates = forget_value."""
    # d c_T / d c_0 = prod_{j=0}^{T-1} diag(f_{j+1}) = diag(f^T)
    # Since all f_t = forget_value, result is diag(forget_value^T)
    # ||diag(f^T)||_F = sqrt(d) * forget_value^T
    fvec = np.full(d, forget_value)
    # Simulate: track gradient product
    grad = np.ones(d)  # represents diagonal of the product
    for _ in range(T):
        grad *= fvec
    return la.norm(grad)

forget_values = [0.0, 0.5, 0.9, 0.99, 1.0]
print(f'Gradient ||d c_T / d c_0||_F after T={T5} steps:')
print(f'{"Forget gate":>15} {"Gradient norm":>15} {"Interpretation":>25}')
print('-' * 60)
for fv in forget_values:
    norm = cell_grad_through_forget(fv, T5, d5)
    if fv == 0.0:
        interp = 'Complete reset'
    elif fv < 0.5:
        interp = 'Strong forgetting'
    elif fv < 0.99:
        interp = 'Partial memory'
    else:
        interp = 'Near-perfect memory'
    print(f'{fv:>15.2f} {norm:>15.4f} {interp:>25}')

print('\nContrast: vanilla RNN gradient norm after T=20 steps')
for rho in [0.5, 0.9, 1.0]:
    norm_rnn = rho ** T5 * np.sqrt(d5)  # rough bound
    print(f'  rho(W_h)={rho}: ~{norm_rnn:.4f}')

In [ ]:
# === 4.5 LSTM vs RNN Parameter Count ===

def count_params(arch, p, d, k):
    if arch == 'RNN':
        return d*d + d*p + d + k*d + k
    elif arch == 'LSTM':
        return 4*d*(d+p) + 4*d + k*d + k
    elif arch == 'GRU':
        return 3*d*(d+p) + 3*d + k*d + k

print(f'{"Architecture":<10} {"p":>6} {"d":>6} {"k":>8} {"Params":>12} {"Ratio vs RNN":>14}')
print('-' * 65)
for (p5, d5, k5) in [(64, 256, 1000), (512, 512, 30000)]:
    rnn_p = count_params('RNN', p5, d5, k5)
    for arch in ['RNN', 'LSTM', 'GRU']:
        params = count_params(arch, p5, d5, k5)
        ratio = params / rnn_p
        print(f'{arch:<10} {p5:>6} {d5:>6} {k5:>8} {params:>12,} {ratio:>13.2f}x')
    print()

print('Key: LSTM ~ 4x RNN (4 gates), GRU ~ 3x RNN (3 gate equations)')

---

## 5. Gated Recurrent Units (GRU)

The GRU merges cell and hidden state into one $\mathbf{h}_t$, using two gates:
$$\mathbf{z}_t = \sigma(W_z [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_z) \quad\text{(update gate)}$$
$$\mathbf{r}_t = \sigma(W_r [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_r) \quad\text{(reset gate)}$$
$$\tilde{\mathbf{h}}_t = \tanh(W_h [\mathbf{r}_t \odot \mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_h)$$
$$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$$

In [ ]:
# === 5.1 GRU Forward Pass ===

class GRUCell:
    def __init__(self, input_dim, hidden_dim, seed=0):
        rng = np.random.default_rng(seed)
        n = hidden_dim + input_dim
        scale = 1.0 / np.sqrt(n)
        self.Wz = rng.standard_normal((hidden_dim, n)) * scale
        self.Wr = rng.standard_normal((hidden_dim, n)) * scale
        self.Wh = rng.standard_normal((hidden_dim, n)) * scale
        self.bz = np.zeros(hidden_dim)
        self.br = np.zeros(hidden_dim)
        self.bh = np.zeros(hidden_dim)
        self.d = hidden_dim

    def step(self, h_prev, x):
        inp = np.concatenate([h_prev, x])
        z = 1 / (1 + np.exp(-(self.Wz @ inp + self.bz)))  # update gate
        r = 1 / (1 + np.exp(-(self.Wr @ inp + self.br)))  # reset gate
        h_candidate = np.tanh(self.Wh @ np.concatenate([r * h_prev, x]) + self.bh)
        h = (1 - z) * h_prev + z * h_candidate
        return h, {'z': z, 'r': r, 'h_cand': h_candidate}

    def forward(self, xs):
        T = len(xs)
        h = np.zeros(self.d)
        hs, gates_list = [], []
        for t in range(T):
            h, g = self.step(h, xs[t])
            hs.append(h.copy()); gates_list.append(g)
        return np.array(hs), gates_list

import numpy as np
import numpy.linalg as la
try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
np.random.seed(42)

p6, d6, T6 = 8, 12, 15
gru = GRUCell(p6, d6)
xs6 = np.random.randn(T6, p6)
hs6, gs6 = gru.forward(xs6)

print(f'GRU output shape: {hs6.shape}')
print(f'Update gate range: [{np.array([g["z"] for g in gs6]).min():.3f}, '
      f'{np.array([g["z"] for g in gs6]).max():.3f}]')

# Verify: z=0 means copy previous state
gru_test = GRUCell(2, 4)
h_prev = np.array([0.5, -0.3, 0.1, 0.7])
x_test = np.ones(2)
# Force z=0 by setting Wz very negative
gru_test.Wz -= 100
h_out, g = gru_test.step(h_prev, x_test)
print(f'\nWith z~0: h_out ≈ h_prev?  {np.allclose(h_out, h_prev, atol=0.01)}')
print(f'  z values: {g["z"][:4].round(4)}  (near 0)')
print('PASS — update gate z=0 copies previous state')

---

## 6. Stability Analysis and Dynamical Systems View

The spectral radius $\rho(W_h) = \max_i |\lambda_i(W_h)|$ determines the memory timescale:
- $\rho < 1$: stable (perturbations decay), short memory
- $\rho \approx 1$: **edge of chaos** — maximal memory, ideal for computation
- $\rho > 1$: unstable (perturbations grow), chaotic

In [ ]:
# === 6.2 Spectral Radius and Memory Horizon ===

def effective_memory_horizon(Wh, threshold=0.01):
    """Largest k such that ||Wh^k||_F > threshold."""
    d = Wh.shape[0]
    J = np.eye(d)
    for k in range(1, 200):
        J = Wh @ J
        if la.norm(J, 'fro') < threshold:
            return k - 1
    return 200

print('Spectral radius vs effective memory horizon (||Wh^k||_F < 0.01):')
print(f'{"rho":>8} {"horizon (steps)":>20} {"Interpretation":>30}')
print('-' * 65)

for rho in [0.5, 0.7, 0.9, 0.95, 0.99, 1.0]:
    Q7, _ = la.qr(np.random.randn(20, 20))
    Wh7 = rho * Q7
    horizon = effective_memory_horizon(Wh7)
    if rho < 0.7:
        interp = 'Very short (local patterns only)'
    elif rho < 0.9:
        interp = 'Short-medium range'
    elif rho < 0.99:
        interp = 'Medium-long range'
    else:
        interp = 'Very long / permanent'
    print(f'{rho:>8.2f} {horizon:>20} {interp:>30}')

# Analytical prediction: horizon ≈ log(0.01/sqrt(d)) / log(rho)
print('\nAnalytical: horizon ≈ log(0.01/√d) / log(ρ) for ρ < 1')
d = 20
for rho in [0.5, 0.9, 0.95]:
    import math
    h_analytical = math.log(0.01 / d**0.5) / math.log(rho)
    print(f'  rho={rho}: analytical ≈ {h_analytical:.0f} steps')

In [ ]:
# === 6.4 Orthogonal Initialisation ===
# Orthogonal W_h: rho = 1 exactly, ||Wh^T||_2 = 1 for all T

def random_orthogonal(d, seed=42):
    rng = np.random.default_rng(seed)
    Q, _ = la.qr(rng.standard_normal((d, d)))
    return Q

d8 = 10
Wh_ortho = random_orthogonal(d8)
Wh_gauss = np.random.randn(d8, d8) / np.sqrt(d8)

print('Orthogonal W_h verification:')
print(f'  W^T W == I:  {np.allclose(Wh_ortho.T @ Wh_ortho, np.eye(d8), atol=1e-10)}')
print(f'  rho(W_ortho) = {max(abs(la.eigvals(Wh_ortho))):.6f}  (should be 1.0)')
print(f'  rho(W_gauss) = {max(abs(la.eigvals(Wh_gauss))):.6f}  (random, often ~1.0 by circular law)')

# Gradient norm over T steps: orthogonal vs random Gaussian
T_test = 50
norms_orth = []
norms_gauss = []
Jo, Jg = np.eye(d8), np.eye(d8)
for t in range(T_test):
    Jo = Wh_ortho @ Jo
    Jg = Wh_gauss @ Jg
    norms_orth.append(la.norm(Jo, 2))
    norms_gauss.append(la.norm(Jg, 2))

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(norms_orth, label='Orthogonal W_h', color='#2ca02c', linewidth=2)
    ax.plot(norms_gauss, label='Gaussian W_h (1/√d)', color='#d62728', linewidth=2)
    ax.axhline(1.0, color='black', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time steps T'); ax.set_ylabel('||W_h^T||_2')
    ax.set_title('Spectral Norm of W_h^T: Orthogonal vs Gaussian Init')
    ax.legend()
    plt.tight_layout(); plt.show()

print(f'\nSpectral norm ||W_ortho^{T_test}||_2 = {norms_orth[-1]:.4f}  (should stay near 1.0)')
print(f'Spectral norm ||W_gauss^{T_test}||_2  = {norms_gauss[-1]:.4f}')

---

## 7. Initialisation and Training Strategies

Three key training insights for RNNs/LSTMs:
1. **Forget gate bias = 1**: ensures LSTM starts with memory mostly preserved
2. **Orthogonal init for $W_h$**: ensures $\rho = 1$ at start of training
3. **Gradient clipping**: prevents explosion, allows larger learning rates

In [ ]:
# === 7.1 Forget Gate Bias Effect ===
# Compare b_f = 0 vs b_f = 1 on gradient flow

def lstm_cell_grad_norm(b_forget, T, d, seed=0):
    """Simulate gradient norm through cell state with constant forget gate."""
    # With b_f = b_forget and random input, average forget gate value
    avg_f = 1 / (1 + np.exp(-b_forget))  # sigmoid of bias (zero input component)
    # Gradient through cell state after T steps: product of forget gates
    # Using avg_f as approximation
    grad_norm = avg_f ** T * np.sqrt(d)
    return avg_f, grad_norm

T9 = 30
d9 = 16
print(f'Effect of forget gate bias on gradient flow (T={T9} steps, d={d9}):')
print(f'{"b_f":>6} {"avg f_t":>10} {"||grad||":>12} {"Verdict":>25}')
print('-' * 60)
for bf in [-2.0, -1.0, 0.0, 0.5, 1.0, 2.0]:
    avg_f, gnorm = lstm_cell_grad_norm(bf, T9, d9)
    verdict = 'OK' if gnorm > 0.1 else 'VANISHED' if gnorm < 0.001 else 'WEAK'
    print(f'{bf:>6.1f} {avg_f:>10.3f} {gnorm:>12.6f} {verdict:>25}')

print('\nRecommendation: initialise b_f = 1.0 for stable training')
print('  This gives avg forget gate ~ 0.73 at init: most memory preserved')

---

## 8. Variants: Bidirectional RNNs and Bahdanau Attention

**Bahdanau attention** (2015) solves the fixed-bottleneck problem by letting the decoder attend to all encoder hidden states:
$$e_{t,s} = \mathbf{v}^\top \tanh(W_a \mathbf{h}_s^{(\text{enc})} + U_a \mathbf{h}_{t-1}^{(\text{dec})})$$
$$\alpha_{t,s} = \text{softmax}_s(e_{t,s}), \quad \mathbf{c}_t = \sum_s \alpha_{t,s} \mathbf{h}_s^{(\text{enc})}$$

In [ ]:
# === 8.4 Bahdanau Attention ===

class BahdanauAttention:
    def __init__(self, enc_dim, dec_dim, attn_dim, seed=7):
        rng = np.random.default_rng(seed)
        scale = 0.1
        self.Wa = rng.standard_normal((attn_dim, enc_dim)) * scale
        self.Ua = rng.standard_normal((attn_dim, dec_dim)) * scale
        self.v  = rng.standard_normal(attn_dim) * scale

    def score(self, enc_states, dec_h):
        """Compute attention scores e_{t,s} for all s.
        enc_states: (T_x, enc_dim)
        dec_h: (dec_dim,)
        Returns: (T_x,)
        """
        # enc_states @ Wa^T: (T_x, attn_dim)
        enc_proj = enc_states @ self.Wa.T
        dec_proj = dec_h @ self.Ua.T  # (attn_dim,)
        e = np.tanh(enc_proj + dec_proj) @ self.v  # (T_x,)
        return e

    def attend(self, enc_states, dec_h):
        """Returns context vector and attention weights."""
        e = self.score(enc_states, dec_h)
        alpha = np.exp(e - e.max()); alpha /= alpha.sum()  # stable softmax
        context = alpha @ enc_states  # (enc_dim,)
        return context, alpha

# Demo: simple seq2seq-style attention
T_x = 8   # source length
T_y = 5   # target length
enc_d, dec_d, attn_d = 16, 16, 8

attn = BahdanauAttention(enc_d, dec_d, attn_d)
enc_states = np.random.randn(T_x, enc_d)

# Simulate decoder steps
alpha_matrix = np.zeros((T_y, T_x))
dec_h = np.zeros(dec_d)
for t in range(T_y):
    ctx, alpha = attn.attend(enc_states, dec_h)
    alpha_matrix[t] = alpha
    dec_h = np.tanh(ctx + dec_h)  # simplified decoder step

print(f'Attention weight matrix shape: {alpha_matrix.shape}  (T_y x T_x)')
print(f'Each row sums to 1: {np.allclose(alpha_matrix.sum(axis=1), 1.0)}')
print('PASS — attention weights are valid probability distributions')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(alpha_matrix, cmap='viridis', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('Encoder position (source)')
    ax.set_ylabel('Decoder step (target)')
    ax.set_title('Bahdanau Attention Weight Matrix alpha_{t,s}')
    ax.set_xticks(range(T_x)); ax.set_yticks(range(T_y))
    plt.tight_layout(); plt.show()
    print('Heatmap: brighter = more attention; row = decoder step, col = encoder position')

---

## 9. Applications: Language Modelling and Mamba Connection

**Perplexity** measures language model quality:
$$\text{PP} = \exp\!\left(-\frac{1}{T}\sum_{t=1}^T \log P(x_t \mid x_{1:t-1})\right)$$

Lower is better. A perplexity of $k$ means the model is as uncertain as choosing uniformly over $k$ options.

In [ ]:
# === 9.1 Character-Level Language Model: Perplexity ===

def compute_perplexity(model_forward, text_ids, vocab_size):
    """Compute perplexity of a model on a sequence of token IDs."""
    T = len(text_ids) - 1  # predict each token from the previous
    total_log_prob = 0.0
    h = np.zeros(model_forward.hidden_dim)
    for t in range(T):
        x_t = np.zeros(vocab_size); x_t[text_ids[t]] = 1.0  # one-hot
        h = np.tanh(model_forward.Wh @ h + model_forward.Wx @ x_t + model_forward.bh)
        z = model_forward.Wo @ h + model_forward.bo
        probs = np.exp(z - z.max()); probs /= probs.sum()
        target = text_ids[t + 1]
        total_log_prob += np.log(probs[target] + 1e-10)
    return np.exp(-total_log_prob / T)

# Simulate with a tiny character vocabulary
vocab = 'abcdefghijklmnopqrstuvwxyz '
V = len(vocab)
char2id = {c: i for i, c in enumerate(vocab)}

text = 'hello world this is a test sentence for perplexity'
text_ids = [char2id[c] for c in text if c in char2id]

# Untrained RNN (random weights) — should have high perplexity
rnn_lm = VanillaRNN(V, 32, V, seed=1)
pp = compute_perplexity(rnn_lm, text_ids, V)
print(f'Untrained RNN perplexity: {pp:.1f}')
print(f'  (Random baseline: ~{V} = {V}, higher means worse than random)')
print(f'  Typical trained char-RNN PPL: ~3-5 on short text')
print(f'  Word-level LSTM on PTB: ~58-65 PPL')
print(f'  GPT-3 on PTB equivalent: ~20-25 PPL')

# Teacher forcing illustration
print('\nTeacher forcing vs free-running:')
print('  Training:   feed ground-truth x_{t-1} at step t  (stable, fast convergence)')
print('  Inference:  feed predicted x̂_{t-1} at step t  (may accumulate errors)')
print('  Fix:        scheduled sampling — gradually switch during training')

# Perplexity = exp(cross-entropy)
ce = np.log(pp)
print(f'\nPerplexity = exp(CE): exp({ce:.2f}) = {np.exp(ce):.2f} ✓')

In [ ]:
# === 9.4 Mamba / SSM Connection ===
# Discrete SSM: h_t = A * h_{t-1} + B * x_t, y_t = C * h_t
# This is a LINEAR RNN — but with structured A for long-range memory

def linear_ssm_forward(A, B, C, xs):
    """Simulate a discrete linear state space model."""
    N = A.shape[0]
    h = np.zeros(N)
    ys = []
    for x in xs:
        h = A @ h + B * x  # B is (N,) for scalar input
        y = C @ h
        ys.append(y)
    return np.array(ys)

# HiPPO-LegS matrix (simplified 4x4) — approximates polynomial history
# Real HiPPO uses specific A_{nk} = sqrt(2n+1)*sqrt(2k+1) for n>k
N_ssm = 4
A_hippo = np.zeros((N_ssm, N_ssm))
for n in range(N_ssm):
    for k in range(n):
        A_hippo[n, k] = -np.sqrt((2*n+1) * (2*k+1))
    A_hippo[n, n] = -(n + 1)
# Discretise with step size dt
dt = 0.01
A_disc = np.eye(N_ssm) + dt * A_hippo  # Euler discretisation
B_disc = dt * np.ones(N_ssm)
C_disc = np.ones(N_ssm)

# Test: ramp signal
T_ssm = 50
xs_ssm = np.linspace(0, 1, T_ssm)
ys_ssm = linear_ssm_forward(A_disc, B_disc, C_disc, xs_ssm)

print('Linear SSM (HiPPO-inspired) demo:')
print(f'  Input: ramp from 0 to 1 over {T_ssm} steps')
print(f'  Output at T={T_ssm}: {ys_ssm[-1]:.4f}')
print(f'  State dim: {N_ssm} (vs LSTM: {d6})')

print('\nLSTM <-> Mamba correspondence:')
print('  LSTM forget gate f_t  <->  Mamba step size Δ_t (input-dependent)')
print('  LSTM cell state c_t   <->  Mamba state h_t')
print('  LSTM input gate i_t   <->  Mamba input projection B_t')
print('  LSTM output gate o_t  <->  Mamba output projection C_t')
print('  Both: gating controls what information to keep vs discard')
print('  Mamba: hardware-efficient parallel scan; O(T) not O(T^2)')

---

## Summary

| Concept | Formula | Key Insight |
|---|---|---|
| Vanilla RNN | $\mathbf{h}_t = \sigma(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b})$ | Shared weights across time |
| BPTT gradient | $\prod_{j=k}^{T-1} D_j W_h$ | Jacobian product → vanishing |
| LSTM cell update | $\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$ | Additive = gradient highway |
| GRU update | $\mathbf{h}_t = (1-\mathbf{z}_t)\odot\mathbf{h}_{t-1} + \mathbf{z}_t\odot\tilde{\mathbf{h}}_t$ | Interpolation |
| Stability | $\rho(W_h) < 1$ ↔ stable; $\rho \approx 1$ = edge of chaos | Spectral radius controls memory |
| Bahdanau attn | $\alpha_{t,s} = \text{softmax}(\mathbf{v}^\top \tanh(W_a\mathbf{h}_s + U_a\mathbf{h}_{t-1}))$ | Solves bottleneck |
| Mamba | $h_t = A_t h_{t-1} + B_t x_t$ (input-selective) | LSTM gating + SSM efficiency |

**Next:** [Transformer Architecture](../05-Transformer-Architecture/notes.md) — attention replaces recurrence entirely.